In [ ]:
# ============================================================
# LoRA Rank Ablation Study — Full Colab Notebook
# Copy each section between === CELL === markers into a new
# Colab cell. Runtime: T4 GPU (Runtime > Change runtime type)
# ============================================================

# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 1 — Runtime check (run this first, always)        ║
# ╚══════════════════════════════════════════════════════════╝
import torch

if not torch.cuda.is_available():
    raise SystemError(
        "No GPU detected!\n"
        "Go to: Runtime > Change runtime type > T4 GPU > Save\n"
        "Then run this cell again."
    )

gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU : {gpu}")
print(f"VRAM: {vram:.1f} GB")
print("Ready to go!")






In [ ]:
 #╔══════════════════════════════════════════════════════════╗
# ║  CELL 2 — Install all dependencies                      ║
# ║  Takes ~3-4 minutes. Run once per session.              ║
# ╚══════════════════════════════════════════════════════════╝

# Unsloth: dramatically reduces VRAM usage on T4
# It patches flash-attention and 4-bit quantization kernels
# so Phi-3-mini (3.8B) fits comfortably in 15GB VRAM
get_ipython().system('pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"')
get_ipython().system('pip install -q wandb datasets trl peft transformers accelerate bitsandbytes')
get_ipython().system('pip install -q matplotlib pandas scipy')

print("All packages installed.")

In [ ]:

# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 3 — Config (edit these values freely)             ║
# ╚══════════════════════════════════════════════════════════╝

# ── Model ────────────────────────────────────────────────────
BASE_MODEL      = "unsloth/Phi-3-mini-4k-instruct"   # 3.8B, fits T4 in 4-bit
MAX_SEQ_LENGTH  = 1024      # reduce if OOM; 2048 needs more VRAM

# ── LoRA ranks to test ───────────────────────────────────────
# Each rank is one full training run. Start with fewer if testing.
RANKS           = [4, 8, 16, 32, 64]

# ── Training ─────────────────────────────────────────────────
TRAIN_SAMPLES   = 2000      # samples from Alpaca (full = 52K, very slow)
EVAL_SAMPLES    = 200       # held-out evaluation samples
EPOCHS          = 1         # 1 epoch per rank to keep runtime under 12h
BATCH_SIZE      = 2
GRAD_ACCUM      = 4         # effective batch = BATCH_SIZE * GRAD_ACCUM = 8
LR              = 2e-4

# ── LoRA targets ─────────────────────────────────────────────
# These are the attention projection matrices in Phi-3
# More modules = more expressive but slower
LORA_TARGETS    = ["q_proj", "k_proj", "v_proj", "o_proj"]

# ── Checkpoint directory (Google Drive = survives session end) ──
CHECKPOINT_DIR  = "/content/drive/MyDrive/lora_ablation_checkpoints"
USE_DRIVE       = True      # set False if you don't want to mount Drive

# ── W&B (optional but recommended) ───────────────────────────
USE_WANDB       = True      # set False to skip W&B logging
WANDB_PROJECT   = "lora-rank-ablation"

print("Config loaded.")
for k, v in {
    "Base model": BASE_MODEL, "Ranks": RANKS,
    "Train samples": TRAIN_SAMPLES, "Epochs": EPOCHS,
    "Checkpoint dir": CHECKPOINT_DIR
}.items():
    print(f"  {k}: {v}")


In [ ]:


# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 4 — Mount Google Drive for checkpoints           ║
# ║  This lets you resume if Colab disconnects             ║
# ╚══════════════════════════════════════════════════════════╝
import os

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f"Drive mounted. Checkpoints will save to:\n  {CHECKPOINT_DIR}")
else:
    CHECKPOINT_DIR = "/content/lora_ablation_checkpoints"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f"Using local storage (lost on disconnect):\n  {CHECKPOINT_DIR}")




In [ ]:

# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 5 — W&B login (skip if USE_WANDB = False)        ║
# ╚══════════════════════════════════════════════════════════╝
if USE_WANDB:
    import wandb
    # Paste your W&B API key when prompted
    # Get it free at: wandb.ai → User Settings → API Keys
    wandb.login()
    print("W&B logged in.")
else:
    print("Skipping W&B.")



In [ ]:

# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 6 — Load and prepare dataset                     ║
# ║  Alpaca Cleaned: 52K instruction-following examples    ║
# ╚══════════════════════════════════════════════════════════╝
from datasets import load_dataset

print("Loading Alpaca Cleaned dataset...")
ds = load_dataset("yahma/alpaca-cleaned", split="train")

# Alpaca format: instruction + optional input + output
# We format it into a single prompt string that the model learns to complete
def format_alpaca(row):
    """
    Converts Alpaca's three fields into a single training string.
    The model learns to produce everything after 'Response:'.
    """
    if row["input"]:
        prompt = (
            f"### Instruction:\n{row['instruction']}\n\n"
            f"### Input:\n{row['input']}\n\n"
            f"### Response:\n{row['output']}"
        )
    else:
        prompt = (
            f"### Instruction:\n{row['instruction']}\n\n"
            f"### Response:\n{row['output']}"
        )
    return {"text": prompt}

ds = ds.map(format_alpaca, remove_columns=ds.column_names)

# Split: first TRAIN_SAMPLES for training, next EVAL_SAMPLES for evaluation
train_ds = ds.select(range(TRAIN_SAMPLES))
eval_ds  = ds.select(range(TRAIN_SAMPLES, TRAIN_SAMPLES + EVAL_SAMPLES))

print(f"Train: {len(train_ds)} samples")
print(f"Eval:  {len(eval_ds)} samples")
print("\nSample prompt:")
print(train_ds[0]["text"][:300])


In [ ]:


# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 7 — Helper: check if rank already completed      ║
# ║  This is the resume logic — skips finished ranks       ║
# ╚══════════════════════════════════════════════════════════╝
import json

RESULTS_FILE = f"{CHECKPOINT_DIR}/all_results.json"

def load_existing_results():
    """Load results from previous runs (for resuming)."""
    if os.path.exists(RESULTS_FILE):
        with open(RESULTS_FILE) as f:
            results = json.load(f)
        print(f"Loaded existing results for ranks: {list(results.keys())}")
        return results
    return {}

def save_results(results):
    """Save results after each rank completes."""
    with open(RESULTS_FILE, "w") as f:
        json.dump(results, f, indent=2)

def rank_checkpoint_path(rank):
    return f"{CHECKPOINT_DIR}/rank_{rank}"

def rank_already_done(rank, results):
    """True if this rank has saved results AND a checkpoint exists."""
    return (
        str(rank) in results and
        os.path.exists(rank_checkpoint_path(rank))
    )

# Load whatever was already completed
all_results = load_existing_results()
completed   = [int(r) for r in all_results.keys()]
remaining   = [r for r in RANKS if r not in completed]

print(f"\nCompleted ranks : {completed}")
print(f"Remaining ranks : {remaining}")







In [ ]:

# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 8 — Helper: profiling functions                  ║
# ╚══════════════════════════════════════════════════════════╝
import time
import numpy as np

def profile_inference(model, tokenizer, prompt="Explain what a neural network is in simple terms."):
    """
    Measures peak VRAM usage and tokens/sec during inference.
    Returns dict with vram_gb and latency_s.
    """
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    t0     = time.time()

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    latency  = time.time() - t0
    vram_gb  = torch.cuda.max_memory_allocated() / 1e9
    n_tokens = out.shape[1] - inputs["input_ids"].shape[1]

    return {
        "latency_s":    round(latency, 2),
        "tokens_per_s": round(n_tokens / latency, 1),
        "vram_gb":      round(vram_gb, 2),
    }


def compute_perplexity(model, tokenizer, eval_dataset, n_samples=50):
    """
    Perplexity = exp(average negative log-likelihood per token).
    Lower is better — measures how well the model predicts the eval text.
    Uses a subset of eval set to keep it fast.
    """
    from torch.nn import CrossEntropyLoss

    model.eval()
    loss_fn   = CrossEntropyLoss(reduction="mean")
    total_nll = 0.0
    count     = 0

    subset = eval_dataset.select(range(min(n_samples, len(eval_dataset))))

    for row in subset:
        enc = tokenizer(
            row["text"],
            return_tensors="pt",
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
        ).to("cuda")

        input_ids = enc["input_ids"]
        if input_ids.shape[1] < 2:
            continue

        with torch.no_grad():
            outputs = model(**enc, labels=input_ids)
            nll     = outputs.loss.item()

        if not np.isnan(nll) and not np.isinf(nll):
            total_nll += nll
            count     += 1

    model.train()
    avg_nll    = total_nll / max(count, 1)
    perplexity = round(float(np.exp(avg_nll)), 2)
    return perplexity


def count_trainable_params(model):
    """Returns number of trainable parameters in millions."""
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return round(trainable / 1e6, 2)



In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 9 — Main training loop                           ║
# ║                                                        ║
# ║  For each rank:                                        ║
# ║    1. Skip if already done (resume logic)              ║
# ║    2. Load base model fresh                            ║
# ║    3. Attach LoRA adapter at rank r                    ║
# ║    4. Train with SFTTrainer                            ║
# ║    5. Profile VRAM + latency                           ║
# ║    6. Compute perplexity on eval set                   ║
# ║    7. Save checkpoint + results                        ║
# ║    8. Free GPU memory before next rank                 ║
# ╚══════════════════════════════════════════════════════════╝

from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
import gc

for rank in RANKS:

    # ── Resume check ─────────────────────────────────────────
    if rank_already_done(rank, all_results):
        print(f"\n  Rank {rank}: already completed, skipping.")
        continue

    print(f"\n{'='*60}")
    print(f"  Training rank r={rank}")
    print(f"{'='*60}")

    # ── W&B run ──────────────────────────────────────────────
    if USE_WANDB:
        wandb.init(
            project=WANDB_PROJECT,
            name=f"rank-{rank}",
            config={
                "rank": rank, "lora_alpha": rank * 2,
                "base_model": BASE_MODEL,
                "train_samples": TRAIN_SAMPLES, "epochs": EPOCHS,
            },
            reinit=True,
        )

    # ── Step 1: Load base model fresh for each rank ──────────
    # We reload every time so each rank starts from the same
    # untrained weights — a fair comparison
    print(f"  Loading {BASE_MODEL} in 4-bit (QLoRA base)...")
    base_model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,          # auto-detect: bf16 on T4
        load_in_4bit=True,   # NF4 quantization — saves ~60% VRAM
    )
    tokenizer.pad_token     = tokenizer.eos_token
    tokenizer.padding_side  = "right"

    # ── Step 2: Attach LoRA adapter ──────────────────────────
    # lora_alpha = rank * 2 is the standard heuristic
    # Higher alpha = stronger LoRA update relative to frozen weights
    print(f"  Attaching LoRA adapter: rank={rank}, alpha={rank*2}")
    model = FastLanguageModel.get_peft_model(
        base_model,
        r=rank,
        lora_alpha=rank * 2,
        lora_dropout=0.05,
        target_modules=LORA_TARGETS,
        bias="none",
        use_gradient_checkpointing="unsloth",  # saves VRAM during backprop
        random_state=42,
    )

    trainable = count_trainable_params(model)
    print(f"  Trainable parameters: {trainable}M")

    # ── Step 3: Define trainer ────────────────────────────────
    ckpt_path = rank_checkpoint_path(rank)

    training_args = SFTConfig(
        output_dir=ckpt_path,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        save_strategy="epoch",           # save checkpoint at end of epoch
        save_total_limit=1,              # only keep latest checkpoint
        load_best_model_at_end=False,
        warmup_ratio=0.05,
        lr_scheduler_type="cosine",
        weight_decay=0.01,               # L2 regularization
        report_to="wandb" if USE_WANDB else "none",
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        packing=True,                    # pack short sequences together for efficiency
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_ds,
        args=training_args,
    )

    # ── Step 4: Train ─────────────────────────────────────────
    print(f"  Training for {EPOCHS} epoch(s)...")
    train_result = trainer.train()
    train_loss   = round(train_result.training_loss, 4)
    print(f"  Training loss: {train_loss}")

    # ── Step 5: Profile VRAM + latency ───────────────────────
    print("  Profiling inference...")
    FastLanguageModel.for_inference(model)   # unsloth inference mode
    profile = profile_inference(model, tokenizer)
    print(f"  VRAM: {profile['vram_gb']}GB  |  Latency: {profile['latency_s']}s  |  {profile['tokens_per_s']} tok/s")

    # ── Step 6: Compute perplexity ────────────────────────────
    print("  Computing perplexity on eval set (50 samples)...")
    FastLanguageModel.for_training(model)    # switch back to train mode for loss computation
    ppl = compute_perplexity(model, tokenizer, eval_ds, n_samples=50)
    print(f"  Perplexity: {ppl}")

    # ── Step 7: Save results ──────────────────────────────────
    all_results[str(rank)] = {
        "rank":            rank,
        "lora_alpha":      rank * 2,
        "trainable_params_M": trainable,
        "train_loss":      train_loss,
        "perplexity":      ppl,
        "vram_gb":         profile["vram_gb"],
        "latency_s":       profile["latency_s"],
        "tokens_per_s":    profile["tokens_per_s"],
    }
    save_results(all_results)
    print(f"  Results saved to {RESULTS_FILE}")

    # Save the LoRA adapter weights separately (small, ~10-50MB)
    adapter_path = f"{CHECKPOINT_DIR}/rank_{rank}_adapter"
    model.save_pretrained(adapter_path)
    tokenizer.save_pretrained(adapter_path)
    print(f"  Adapter saved to {adapter_path}")

    if USE_WANDB:
        wandb.log(all_results[str(rank)])
        wandb.finish()

    # ── Step 8: Free GPU memory before next rank ─────────────
    # Critical: without this, the next rank OOMs
    del model, base_model, trainer
    gc.collect()
    torch.cuda.empty_cache()
    print(f"  GPU memory freed.")

print("\n" + "="*60)
print("All ranks completed!")
print("="*60)
for r, v in all_results.items():
    print(f"  Rank {r:>3}: perplexity={v['perplexity']:>6.2f}  vram={v['vram_gb']:.1f}GB  loss={v['train_loss']:.4f}")


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 10 — Results table                               ║
# ╚══════════════════════════════════════════════════════════╝
import pandas as pd

rows = []
for r in sorted(all_results.keys(), key=int):
    v = all_results[r]
    rows.append({
        "Rank":              v["rank"],
        "Trainable (M)":     v["trainable_params_M"],
        "Train Loss":        v["train_loss"],
        "Perplexity ↓":      v["perplexity"],
        "VRAM (GB)":         v["vram_gb"],
        "Latency (s)":       v["latency_s"],
        "Tokens/s ↑":        v["tokens_per_s"],
    })

df = pd.DataFrame(rows)
print("\nResults Table:")
print(df.to_string(index=False))

# Highlight best rank (lowest perplexity)
best_rank = df.loc[df["Perplexity ↓"].idxmin(), "Rank"]
print(f"\nBest perplexity at rank: {best_rank}")


In [ ]:

# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 11 — Dashboard visualization                     ║
# ╚══════════════════════════════════════════════════════════╝
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

ranks_done  = sorted([int(r) for r in all_results.keys()])
perplexity  = [all_results[str(r)]["perplexity"]         for r in ranks_done]
vram        = [all_results[str(r)]["vram_gb"]            for r in ranks_done]
loss        = [all_results[str(r)]["train_loss"]         for r in ranks_done]
latency     = [all_results[str(r)]["latency_s"]          for r in ranks_done]
params      = [all_results[str(r)]["trainable_params_M"] for r in ranks_done]

fig = plt.figure(figsize=(18, 10))
fig.suptitle("LoRA Rank Ablation Study — Phi-3-mini on Alpaca", fontsize=16, fontweight="bold")
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# Panel 1: Rank vs Perplexity
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(ranks_done, perplexity, "bo-", linewidth=2, markersize=8)
for x, y in zip(ranks_done, perplexity):
    ax1.annotate(f"{y:.1f}", (x, y), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=9)
ax1.set_xlabel("LoRA Rank (r)")
ax1.set_ylabel("Perplexity (lower = better)")
ax1.set_title("Rank vs Perplexity")
ax1.set_xticks(ranks_done)
ax1.grid(alpha=0.3)

# Panel 2: Rank vs VRAM
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(ranks_done, vram, "ro-", linewidth=2, markersize=8)
for x, y in zip(ranks_done, vram):
    ax2.annotate(f"{y:.1f}GB", (x, y), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=9)
ax2.set_xlabel("LoRA Rank (r)")
ax2.set_ylabel("Peak VRAM (GB)")
ax2.set_title("Rank vs VRAM Usage")
ax2.set_xticks(ranks_done)
ax2.axhline(y=15, color="red", linestyle="--", alpha=0.5, label="T4 limit (15GB)")
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

# Panel 3: VRAM vs Perplexity (Pareto frontier)
ax3 = fig.add_subplot(gs[0, 2])
scatter = ax3.scatter(vram, perplexity, c=ranks_done, cmap="viridis", s=200, zorder=5)
for x, y, r in zip(vram, perplexity, ranks_done):
    ax3.annotate(f"r={r}", (x, y), textcoords="offset points", xytext=(6, 0), fontsize=9)
plt.colorbar(scatter, ax=ax3, label="Rank")
ax3.set_xlabel("Peak VRAM (GB)")
ax3.set_ylabel("Perplexity (lower = better)")
ax3.set_title("VRAM vs Quality (Pareto Frontier)")
ax3.grid(alpha=0.3)

# Annotate Pareto optimal point
best_idx = np.argmin(perplexity)
ax3.annotate(
    "Pareto\noptimal",
    (vram[best_idx], perplexity[best_idx]),
    textcoords="offset points", xytext=(-50, -30),
    arrowprops=dict(arrowstyle="->", color="green"),
    color="green", fontsize=9
)

# Panel 4: Rank vs Training Loss
ax4 = fig.add_subplot(gs[1, 0])
ax4.bar(ranks_done, loss, color="#4C72B0", alpha=0.8, width=3)
for x, y in zip(ranks_done, loss):
    ax4.text(x, y + 0.001, f"{y:.4f}", ha="center", va="bottom", fontsize=9)
ax4.set_xlabel("LoRA Rank (r)")
ax4.set_ylabel("Training Loss")
ax4.set_title("Rank vs Training Loss")
ax4.set_xticks(ranks_done)
ax4.grid(axis="y", alpha=0.3)

# Panel 5: Rank vs Latency
ax5 = fig.add_subplot(gs[1, 1])
ax5.plot(ranks_done, latency, "gs-", linewidth=2, markersize=8)
for x, y in zip(ranks_done, latency):
    ax5.annotate(f"{y:.2f}s", (x, y), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=9)
ax5.set_xlabel("LoRA Rank (r)")
ax5.set_ylabel("Inference Latency (s / 100 tokens)")
ax5.set_title("Rank vs Inference Latency")
ax5.set_xticks(ranks_done)
ax5.grid(alpha=0.3)

# Panel 6: Trainable parameters
ax6 = fig.add_subplot(gs[1, 2])
ax6.bar(ranks_done, params, color="#55A868", alpha=0.8, width=3)
for x, y in zip(ranks_done, params):
    ax6.text(x, y + 0.05, f"{y}M", ha="center", va="bottom", fontsize=9)
ax6.set_xlabel("LoRA Rank (r)")
ax6.set_ylabel("Trainable Parameters (M)")
ax6.set_title("Rank vs Trainable Parameters")
ax6.set_xticks(ranks_done)
ax6.grid(axis="y", alpha=0.3)

plt.savefig(f"{CHECKPOINT_DIR}/ablation_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Dashboard saved to {CHECKPOINT_DIR}/ablation_dashboard.png")




In [ ]:
# CELL 12 — Key findings (fixed variable name clash)
import numpy as np

ranks_arr   = np.array(ranks_done)
ppl_arr     = np.array(perplexity)    # this is the LIST named perplexity
vram_arr    = np.array(vram)          # this is the LIST named vram

# Pareto: best quality per GB of VRAM
efficiency  = ppl_arr / vram_arr
pareto_idx  = np.argmin(efficiency)
pareto_rank = ranks_done[pareto_idx]

# Diminishing returns: where drop between consecutive ranks < 5%
ppl_drops       = np.diff(ppl_arr) / ppl_arr[:-1] * 100
dim_return_idx  = next(
    (i for i, drop in enumerate(ppl_drops) if abs(drop) < 5),
    len(ranks_done) - 1
)
dim_return_rank = ranks_done[dim_return_idx + 1]

# FIX: use ppl_arr (the array) not ppl (the float from training loop)
vram_delta = vram_arr[-1] - vram_arr[0]
ppl_delta  = ppl_arr[0]  - ppl_arr[-1]

print("=" * 55)
print("KEY FINDINGS")
print("=" * 55)
print(f"  Pareto optimal rank     : r={pareto_rank}")
print(f"  Diminishing returns at  : r={dim_return_rank}")
print(f"  VRAM cost r=4 -> r=64   : +{vram_delta:.1f} GB")
print(f"  Perplexity gain r=4->r=64: {ppl_delta:.3f} points")
print(f"  Recommendation          : r={pareto_rank}")
print("=" * 55)